In [ ]:
# ============================================================
# WiDS 2026 — FULL PIPELINE (SINGLE CELL, COLAB SAFE)
# ============================================================
!pip install -q scikit-survival lightgbm

import numpy as np
import pandas as pd
import warnings
import itertools
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from sksurv.util import Surv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─── CONFIG ───────────────────────────────────────────────────
HORIZONS   = [12, 24, 48, 72]
N_FOLDS    = 10
GBSA_SEEDS = [42, 123, 456, 789]
LGB_SEEDS  = [42, 123, 456, 789, 777]

GBSA_CONFIGS = [
    {"n_estimators": 1000, "learning_rate": 0.015, "max_depth": 3, "min_samples_leaf": 15, "subsample": 0.80},
    {"n_estimators": 1200, "learning_rate": 0.010, "max_depth": 2, "min_samples_leaf": 25, "subsample": 0.85},
    {"n_estimators": 1500, "learning_rate": 0.008, "max_depth": 3, "min_samples_leaf": 20, "subsample": 0.75},
]

LGB_PARAMS = {
    12: {"n_estimators": 300, "learning_rate": 0.025, "max_depth": 3, "num_leaves": 7,
         "min_child_samples": 10, "subsample": 0.70, "colsample_bytree": 0.70,
         "reg_alpha": 0.5, "reg_lambda": 2.0},
    24: {"n_estimators": 300, "learning_rate": 0.025, "max_depth": 3, "num_leaves": 7,
         "min_child_samples": 10, "subsample": 0.70, "colsample_bytree": 0.70,
         "reg_alpha": 0.5, "reg_lambda": 2.0},
    48: {"n_estimators": 400, "learning_rate": 0.020, "max_depth": 4, "num_leaves": 15,
         "min_child_samples": 8,  "subsample": 0.70, "colsample_bytree": 0.70,
         "reg_alpha": 1.0, "reg_lambda": 3.0},
}

print("✅ Config ready!")

# ─── LOAD DATA ────────────────────────────────────────────────
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
print(f"Train: {train.shape} | Events: {train['event'].value_counts().to_dict()}")
print(f"Test : {test.shape}")

# ─── FEATURE ENGINEERING ──────────────────────────────────────
def engineer_features(df, train_medians=None):
    df = df.copy()
    num_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c not in ["event_id", "event", "time_to_hit_hours"]]
    if train_medians is None:
        train_medians = df[num_cols].median()
    df[num_cols] = df[num_cols].fillna(train_medians)

    dist   = df["dist_min_ci_0_5h"].clip(lower=1)
    speed  = df["closing_speed_m_per_h"]
    perims = df["num_perimeters_0_5h"]
    af     = df["area_first_ha"]
    radg   = df["radial_growth_rate_m_per_h"].clip(lower=0)
    al     = df["alignment_abs"]
    cp     = speed.clip(lower=0)
    ec     = cp + radg
    gr     = df["area_growth_rate_ha_per_h"]

    df["log_distance"]    = np.log1p(dist)
    df["inv_distance"]    = 1 / (dist / 1000 + 0.1)
    df["inv_distance_sq"] = df["inv_distance"] ** 2
    df["dist_rank"]       = dist.rank(pct=True)
    df["fire_radius_km"]  = np.sqrt(af * 10000 / np.pi) / 1000
    df["radius_to_dist"]  = (df["fire_radius_km"] * 1000) / dist
    df["log_area_dist"]   = np.log1p(af) - np.log1p(dist)
    df["area_to_dist"]    = af / (dist / 1000 + 0.1)

    df["log_eta"]           = np.log1p(np.where(cp > 0.01, dist / cp, 9999).clip(max=9999))
    df["log_eta_effective"] = np.log1p(np.where(ec > 0.01, dist / ec, 9999).clip(max=9999))
    df["eta_ratio"]         = dist / (ec + 1)

    df["threat_score"]     = al * cp / np.log1p(dist)
    df["threat_sq"]        = df["threat_score"] ** 2
    df["growth_pressure"]  = gr * al
    df["growth_intensity"] = gr * perims
    df["urgency"]          = perims * cp / (dist / 1000 + 0.1)
    df["risk_score"]       = (df["inv_distance"] * cp * al * (1 + radg)) / (df["log_eta"] + 0.1)

    df["log_speed"]        = np.log1p(cp)
    df["speed_squared"]    = cp ** 2
    df["speed_radg_ratio"] = cp / (radg + 1)

    df["dist_growth_interaction"] = gr / (dist + 100)
    df["closing_pressure"]        = ec / (dist + 100)
    df["inv_dist_x_speed"]        = df["inv_distance"] * cp
    df["inv_dist_x_align"]        = df["inv_distance"] * al
    df["speed_x_align"]           = cp * al
    df["perims_x_align"]          = perims * al
    df["threat_x_perims"]         = df["threat_score"] * perims
    df["dist_x_radg"]             = df["log_distance"] * radg
    df["eta_vs_growth"]           = df["log_eta"] * gr
    df["log1p_growth"]            = np.log1p(gr)

    df["growth_sq"]  = gr ** 2
    df["align_sq"]   = al ** 2
    df["perims_sq"]  = perims ** 2
    df["area_rank"]  = af.rank(pct=True)
    df["growth_rank"]= gr.rank(pct=True)

    df["zone_critical"]    = (dist < 5000).astype(float)
    df["zone_warning"]     = ((dist >= 5000) & (dist < 20000)).astype(float)
    df["zone_safe"]        = (dist >= 20000).astype(float)
    df["very_close"]       = (dist < 2000).astype(float)
    df["has_movement"]     = (perims > 1).astype(float)
    df["is_static"]        = ((perims <= 1) | (df["low_temporal_resolution_0_5h"] == 1)).astype(float)
    df["explosive_growth"] = (gr > gr.median()).astype(float)

    lq = df["low_temporal_resolution_0_5h"]
    df["lq_x_dist"]   = lq * df["log_distance"]
    df["lq_x_perims"] = lq * perims

    month = df["event_start_month"]
    hour  = df["event_start_hour"]
    df["month_sin"]       = np.sin(2 * np.pi * month / 12)
    df["month_cos"]       = np.cos(2 * np.pi * month / 12)
    df["hour_sin"]        = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"]        = np.cos(2 * np.pi * hour / 24)
    df["is_summer"]       = month.isin([6, 7, 8]).astype(float)
    df["is_afternoon"]    = ((hour >= 12) & (hour < 20)).astype(float)
    df["summer_x_threat"] = df["is_summer"] * df["threat_score"]

    return df.replace([np.inf, -np.inf], np.nan).fillna(0), train_medians

train, MEDIANS = engineer_features(train)
test,  _       = engineer_features(test, train_medians=MEDIANS)

RAW_COLS = ["alignment_abs", "num_perimeters_0_5h", "dt_first_last_0_5h",
            "low_temporal_resolution_0_5h", "area_first_ha", "log1p_growth"]

FEATS = RAW_COLS + [c for c in train.columns
                    if c not in RAW_COLS + ["event_id", "event", "time_to_hit_hours"]]

X_lgb       = train[FEATS].values
X_test_lgb  = test[FEATS].values
scaler      = StandardScaler()
X_gbsa      = scaler.fit_transform(X_lgb)
X_test_gbsa = scaler.transform(X_test_lgb)

event  = train["event"].values
time   = train["time_to_hit_hours"].values
y_surv = Surv.from_arrays(event=event.astype(bool), time=time)

print(f"✅ Features: {len(FEATS)}")

# ─── HELPERS ──────────────────────────────────────────────────
def get_probs(model, Xi):
    fns = model.predict_survival_function(Xi)
    out = np.zeros((len(fns), 4))
    for i, fn in enumerate(fns):
        out[i] = 1 - fn(np.clip(HORIZONS, fn.x[0], fn.x[-1]))
    return out

def ipcw(t, e, h):
    order   = np.argsort(t)
    ts, es  = t[order], e[order]
    n       = len(t)
    at_risk = n - np.arange(n)
    G       = np.ones(n)
    for i in range(n):
        if at_risk[i] > 0:
            G[i] = 1 - (es[i] == 0) / at_risk[i]
        if i > 0:
            G[i] *= G[i - 1]
    def G_t(val):
        idx = np.searchsorted(ts, val, side="right") - 1
        return max(G[idx], 0.01) if idx >= 0 else 1.0
    w = np.array([
        1 / G_t(tv) if (ev == 1 and tv <= h) else
        1 / G_t(h)  if tv >= h else 1.0
        for tv, ev in zip(t, e)
    ])
    w = np.clip(w, 0, 3)
    return w / np.mean(w)

def isotonic_calibrate(oof_p, oof_y):
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(oof_p, oof_y)
    return ir

def apply_monotonic_pipeline(pred, p24, p48):
    pred = np.clip(pred, 0, 1)
    pred = np.maximum.accumulate(pred, axis=1)
    pred[:, 1] **= p24
    pred[:, 2] **= p48
    pred[:, 1]  = np.maximum(pred[:, 1], pred[:, 0])
    pred[:, 2]  = np.maximum(pred[:, 2], pred[:, 1])
    pred[:, 3]  = 1.0
    return pred

def score_fn(p, t, e):
    from sksurv.metrics import concordance_index_censored, brier_score
    y_ev = Surv.from_arrays(event=e.astype(bool), time=t)
    risk = 0.2 * p[:, 0] + 0.3 * p[:, 1] + 0.5 * p[:, 2]
    try:    c = concordance_index_censored(e.astype(bool), t, risk)[0]
    except: c = 0.5
    b = 0
    for h, wt, ci in zip([24, 48, 72], [0.3, 0.4, 0.3], [1, 2, 3]):
        try:
            _, bs = brier_score(y_ev, y_ev, p[:, ci:ci+1], np.array([h]))
            b += wt * bs[0]
        except: b += wt * 0.25
    return 0.3 * c + 0.7 * (1 - b)

print("✅ Helpers defined!")

# ─── MODEL A: GBSA ────────────────────────────────────────────
print("\nMODEL A: GBSA (3 configs × 4 seeds × 10 folds)")

oof_g  = np.zeros((len(X_gbsa), 4))
test_g = np.zeros((len(X_test_gbsa), 4))

for cfg_i, cfg in enumerate(GBSA_CONFIGS):
    cfg_oof  = np.zeros((len(X_gbsa), 4))
    cfg_test = np.zeros((len(X_test_gbsa), 4))

    for seed in GBSA_SEEDS:
        kf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        seed_oof  = np.zeros((len(X_gbsa), 4))
        seed_test = np.zeros((len(X_test_gbsa), 4))

        for tr, va in kf.split(X_gbsa, event):
            m = GradientBoostingSurvivalAnalysis(**cfg, random_state=seed)
            m.fit(X_gbsa[tr], y_surv[tr])
            seed_oof[va] += get_probs(m, X_gbsa[va])
            seed_test    += get_probs(m, X_test_gbsa) / N_FOLDS

        cfg_oof  += seed_oof  / len(GBSA_SEEDS)
        cfg_test += seed_test / len(GBSA_SEEDS)

    oof_g  += cfg_oof  / len(GBSA_CONFIGS)
    test_g += cfg_test / len(GBSA_CONFIGS)
    print(f"  Config {cfg_i+1}/{len(GBSA_CONFIGS)} done")

gbsa_cal_test = np.zeros_like(test_g)
for j, h in enumerate(HORIZONS[:3]):
    yb   = ((event == 1) & (time <= h)).astype(int)
    mask = ~((event == 0) & (time < h))
    ir   = isotonic_calibrate(oof_g[mask, j], yb[mask])
    gbsa_cal_test[:, j] = ir.predict(test_g[:, j])
gbsa_cal_test[:, 3] = 1.0

print(f"✅ GBSA done | mean@24h={gbsa_cal_test[:,1].mean():.3f}  mean@48h={gbsa_cal_test[:,2].mean():.3f}")

# ─── MODEL B: LIGHTGBM ────────────────────────────────────────
print("\nMODEL B: LightGBM (3 horizons × 5 seeds × 10 folds)")

l_oof  = {h: np.zeros(len(X_lgb))      for h in [12, 24, 48]}
l_test = {h: np.zeros(len(X_test_lgb)) for h in [12, 24, 48]}

for h in [12, 24, 48]:
    yb      = ((event == 1) & (time <= h)).astype(int)
    mask    = ~((event == 0) & (time < h))
    v_idx   = np.where(mask)[0]
    h_oof   = np.zeros(len(X_lgb))
    h_test  = np.zeros(len(X_test_lgb))

    for seed in LGB_SEEDS:
        kf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        seed_oof  = np.zeros(len(X_lgb))
        seed_test = np.zeros(len(X_test_lgb))

        for tr, va in kf.split(X_lgb[mask], yb[mask]):
            ti = v_idx[tr]
            vi = v_idx[va]
            w  = ipcw(time[ti], event[ti], h)

            m = lgb.LGBMClassifier(**LGB_PARAMS[h], random_state=seed,
                                   objective="binary", verbose=-1)
            m.fit(X_lgb[ti], yb[ti], sample_weight=w,
                  eval_set=[(X_lgb[vi], yb[vi])],
                  callbacks=[lgb.early_stopping(40, verbose=False),
                              lgb.log_evaluation(-1)])

            seed_oof[vi] += m.predict_proba(X_lgb[vi])[:, 1]
            seed_test    += m.predict_proba(X_test_lgb)[:, 1] / N_FOLDS

        h_oof  += seed_oof  / len(LGB_SEEDS)
        h_test += seed_test / len(LGB_SEEDS)

    ir        = isotonic_calibrate(h_oof[mask], yb[mask])
    l_oof[h]  = ir.predict(h_oof)
    l_test[h] = ir.predict(h_test)
    print(f"  {h}h done | mean={l_test[h].mean():.3f}")

print("✅ LightGBM done!")

# ─── GRID SEARCH ──────────────────────────────────────────────
print("\nRunning Grid Search...")

w12      = 0.95
w24      = 0.92
w48_grid = np.round(np.arange(0.35, 0.56, 0.02), 3)
p24_grid = np.round(np.arange(0.95, 1.13, 0.02), 3)
p48_grid = np.round(np.arange(0.90, 1.13, 0.02), 3)

results = []
for w48, p24, p48 in itertools.product(w48_grid, p24_grid, p48_grid):
    scores = []
    for seed in LGB_SEEDS:
        kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for tr, va in kf.split(X_lgb, event):
            p = np.zeros((len(va), 4))
            p[:, 0] = w12 * oof_g[va, 0] + (1 - w12) * l_oof[12][va]
            p[:, 1] = w24 * oof_g[va, 1] + (1 - w24) * l_oof[24][va]
            p[:, 2] = w48 * oof_g[va, 2] + (1 - w48) * l_oof[48][va]
            p[:, 3] = 1.0
            p = apply_monotonic_pipeline(p, p24, p48)
            scores.append(score_fn(p, time[va], event[va]))

    mean_s = np.mean(scores)
    std_s  = np.std(scores)
    lcb    = mean_s - 0.5 * std_s
    results.append({"lcb": lcb, "mean": mean_s, "std": std_s, "params": (w48, p24, p48)})

top5 = sorted(results, key=lambda x: x["lcb"], reverse=True)[:5]
print("\nTop 5 parameter sets:")
for i, r in enumerate(top5, 1):
    print(f"  Rank {i}: LCB={r['lcb']:.4f}  mean={r['mean']:.4f}  "
          f"std={r['std']:.4f}  params={r['params']}")

# ─── EXPORT ───────────────────────────────────────────────────
print("\nExporting submissions...")
for rnk, res in enumerate(top5, 1):
    w48, p24, p48 = res["params"]
    fp = np.zeros((len(X_test_lgb), 4))
    fp[:, 0] = w12 * gbsa_cal_test[:, 0] + (1 - w12) * l_test[12]
    fp[:, 1] = w24 * gbsa_cal_test[:, 1] + (1 - w24) * l_test[24]
    fp[:, 2] = w48 * gbsa_cal_test[:, 2] + (1 - w48) * l_test[48]
    fp[:, 3] = 1.0
    fp = apply_monotonic_pipeline(fp, p24, p48)

    assert (fp >= 0).all() and (fp <= 1).all(),  "Out of [0,1]!"
    assert (fp[:, 1] >= fp[:, 0]).all(),          "24h < 12h!"
    assert (fp[:, 2] >= fp[:, 1]).all(),          "48h < 24h!"
    assert (fp[:, 3] == 1.0).all(),               "72h != 1.0!"

    fname = f"submission_rank{rnk}.csv"
    pd.DataFrame({
        "event_id": test["event_id"],
        "prob_12h":  fp[:, 0],
        "prob_24h":  fp[:, 1],
        "prob_48h":  fp[:, 2],
        "prob_72h":  fp[:, 3],
    }).to_csv(fname, index=False)
    print(f"  Rank {rnk} saved → {fname}  "
          f"[12h={fp[:,0].mean():.3f} 24h={fp[:,1].mean():.3f} "
          f"48h={fp[:,2].mean():.3f} 72h={fp[:,3].mean():.3f}]")

print("\n✅ DONE! Downloading...")
from google.colab import files
for rnk in range(1, 6):
    files.download(f"submission_rank{rnk}.csv")

✅ Config ready!
Train: (221, 37) | Events: {0: 152, 1: 69}
Test : (95, 35)
✅ Features: 84
✅ Helpers defined!

MODEL A: GBSA (3 configs × 4 seeds × 10 folds)
  Config 1/3 done
  Config 2/3 done
  Config 3/3 done
✅ GBSA done | mean@24h=0.269  mean@48h=0.284

MODEL B: LightGBM (3 horizons × 5 seeds × 10 folds)
  12h done | mean=0.187
  24h done | mean=0.274
  48h done | mean=0.289
✅ LightGBM done!

Running Grid Search...

Top 5 parameter sets:
  Rank 1: LCB=0.4867  mean=0.4960  std=0.0186  params=(np.float64(0.55), np.float64(0.95), np.float64(0.9))
  Rank 2: LCB=0.4866  mean=0.4959  std=0.0186  params=(np.float64(0.53), np.float64(0.95), np.float64(0.9))
  Rank 3: LCB=0.4865  mean=0.4958  std=0.0186  params=(np.float64(0.51), np.float64(0.95), np.float64(0.9))
  Rank 4: LCB=0.4864  mean=0.4957  std=0.0186  params=(np.float64(0.49), np.float64(0.95), np.float64(0.9))
  Rank 5: LCB=0.4863  mean=0.4956  std=0.0186  params=(np.float64(0.47), np.float64(0.95), np.float64(0.9))

Exporting subm

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>